In [19]:
import os, glob
import numpy as np
import tensorflow as tf
import pandas as pd

## Prepare dataset

In [20]:
CAPTURES = "../dataset/captures"
MAX_LEN  = 100 
N_CH     = 6 
classes = sorted(d for d in os.listdir(CAPTURES)
                 if os.path.isdir(os.path.join(CAPTURES, d)))

class_to_id = {c: i for i, c in enumerate(classes)}

lengths = []
"""
Load and preprocess the gesture recordings to create the dataset for training the MLP.

For each gesture class, the function:
1. Loads the accelerometer and gyroscope measurements from CSV files.
2. Creates a fixed-size sequence using zero-padding or truncation.
3. Flattens the 2D sensor matrix into a 1D feature vector.
4. Assigns a numerical label to identify the gesture.

Finally, X contains the input features and y contains the corresponding gesture labels.
"""

X, y = [], []
for cls in classes:
    for path in glob.glob(os.path.join(CAPTURES, cls, "*.csv")):
        rows = np.loadtxt(path, delimiter=",", skiprows=1, usecols=range(4, 10))
        seq = np.zeros((MAX_LEN, N_CH), dtype=np.float32)   
        n = min(rows.shape[0], MAX_LEN)
        seq[:n] = rows[:n]
        X.append(seq.flatten())          
        y.append(class_to_id[cls])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print("X", X.shape, "classes", len(classes))  

X (492, 600) classes 16


## Analyze dataset

In [21]:
print(X.shape)

(492, 600)


## Hyperparameter tuning 

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import keras_tuner as kt
import numpy as np
import tensorflow as tf

# 1. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# 2. Normalize data
scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)


# 3. Build neural network
def build_model(hp):
    model = tf.keras.Sequential()

    model.add(tf.keras.layers.Input(shape=(600,)))

    for i in range(hp.Int("num_layers", 1, 3)):
        model.add(tf.keras.layers.Dense(
            units=hp.Int(
                f"units_{i}",
                min_value=16,
                max_value=128,
                step=16
            ),
            activation="relu"
        ))

    model.add(tf.keras.layers.Dense(16, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Choice("lr", [1e-2, 1e-3, 1e-4])
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# 4. Configure Keras Tuner
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=10,
    directory="results",
    project_name="gesture_recognition"
)

# 5. Search for the best architecture
tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30
)

# 6. Retrieve best model
best_model = tuner.get_best_models(num_models=1)[0]

# 7. Retrieve best hyperparameters
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best hyperparameters:", best_hp.values)

# 8. Evaluate using test dataset
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)

print("Test accuracy:", test_accuracy)



Reloading Tuner from results/gesture_recognition/tuner0.json
Best hyperparameters: {'num_layers': 1, 'units_0': 80, 'lr': 0.001, 'units_1': 128, 'units_2': 96}
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9596 - loss: 0.5229  
Test accuracy: 0.9595959782600403


/home/jordi/MERIT/IOTSS/omniflux/Train/enviro/lib/python3.14/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


In [23]:
model = tuner.hypermodel.build(best_hp)

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3280 - loss: 2.4620 - val_accuracy: 0.6329 - val_loss: 1.5776
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7611 - loss: 1.1227 - val_accuracy: 0.8228 - val_loss: 1.1242
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8599 - loss: 0.7727 - val_accuracy: 0.8734 - val_loss: 0.9045
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8981 - loss: 0.5864 - val_accuracy: 0.8861 - val_loss: 0.7678
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9331 - loss: 0.4588 - val_accuracy: 0.8861 - val_loss: 0.6820
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9331 - loss: 0.3804 - val_accuracy: 0.8861 - val_loss: 0.6426
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9427 - loss: 0.3247 - val_accuracy: 0.8987 - val_loss: 0.6267
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9427 - loss: 0.2887 - val_accuracy: 0.9114 - val_loss

## Test trained network

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

loss, acc = model.evaluate(X_test, y_test, verbose=0)
y_pred = model.predict(X_test, verbose=0).argmax(1)

print(f"test loss {loss:.4f}  accuracy {acc:.4f}  ({len(y_test)} samples)")
print(classification_report(y_test, y_pred, target_names=classes, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
for i, c in enumerate(classes):
    print(f"{c:28s} acc={cm[i, i] / cm[i].sum():.2f}  n={cm[i].sum()}")

test loss 0.4017  accuracy 0.8889  (99 samples)
                          precision    recall  f1-score   support

          label_BACKWARD      1.000     1.000     1.000         6
            label_CIRCLE      1.000     1.000     1.000         6
label_CLOCKWISE_ROTATION      1.000     1.000     1.000         6
  label_COUNTERCLOCKWISE      1.000     1.000     1.000         6
              label_DOWN      0.500     0.500     0.500         6
           label_FORWARD      0.833     0.833     0.833         6
              label_LEFT      0.667     1.000     0.800         6
         label_PRONATION      0.857     1.000     0.923         6
  label_RADIAL_DEVIATION      1.000     1.000     1.000         6
             label_RIGHT      0.857     1.000     0.923         6
             label_SHAKE      1.000     0.167     0.286         6
        label_SUPINATION      1.000     1.000     1.000         6
   label_ULNAR_DEVIATION      0.857     1.000     0.923         6
                label_UP   

## Architecture grid search (size vs accuracy)

The first Dense layer holds 76.8k of the 86.2k parameters, so input width dominates the size. Grid over temporal average-pooling (`stride` frames -> one) and hidden layer sizes; every model gets the same epochs and the same split. `int8_kb` assumes 1 byte per parameter (the real TFLite size is measured in the next cell).

99 test samples means roughly +/- 3 points standard error, so treat differences under ~5 points as noise.

In [25]:
GRID_STRIDES = [1, 2, 4]                      # average this many frames together
GRID_HIDDEN  = [(128, 64), (64, 32), (32, 16), (16,)]
GRID_EPOCHS  = 80


def fold(Xs, stride):
    """Average-pool groups of `stride` frames; MAX_LEN is divisible by 1/2/4."""
    if stride == 1:
        return Xs
    return Xs.reshape(-1, MAX_LEN // stride, stride, N_CH).mean(2).reshape(len(Xs), -1)


rows, models, splits = [], {}, {}
for stride in GRID_STRIDES:
    Xtr, Xte = fold(X_train, stride), fold(X_test, stride)
    for hidden in GRID_HIDDEN:
        name = "x".join(map(str, hidden))
        m = tf.keras.Sequential(
            [tf.keras.layers.Input((Xtr.shape[1],))]
            + [tf.keras.layers.Dense(h, activation="relu") for h in hidden]
            + [tf.keras.layers.Dense(len(classes), activation="softmax")])
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
        m.fit(Xtr, y_train, batch_size=32, epochs=GRID_EPOCHS, verbose=0)
        acc = m.evaluate(Xte, y_test, verbose=0)[1]
        models[(stride, name)], splits[(stride, name)] = m, (Xtr, Xte)
        rows.append(dict(stride=stride, hidden=name, params=m.count_params(),
                         int8_kb=m.count_params() / 1024, test_acc=round(acc, 4)))

grid = pd.DataFrame(rows).sort_values("params").reset_index(drop=True)
print(grid.to_string(index=False, float_format="{:.3f}".format))

# smallest model within 2 points of the best accuracy on this grid
best_acc = grid.test_acc.max()
pick = grid[grid.test_acc >= best_acc - 0.02].iloc[0]
best_model = models[(pick.stride, pick.hidden)]
best_X_train, best_X_test = splits[(pick.stride, pick.hidden)]
best_model.save("model_best.keras")
print(f"\npicked stride={pick.stride} hidden={pick.hidden}: "
      f"~{pick['int8_kb']:.1f}KB int8, test acc {pick.test_acc:.4f} "
      f"(best on grid {best_acc:.4f})")

 stride hidden  params  int8_kb  test_acc
      4     16    2688    2.625     0.859
      2     16    5088    4.969     0.909
      4  32x16    5632    5.500     0.919
      1     16    9888    9.656     0.879
      2  32x16   10432   10.188     0.909
      4  64x32   12272   11.984     0.899
      1  32x16   20032   19.562     0.889
      2  64x32   21872   21.359     0.899
      4 128x64   28624   27.953     0.899
      1  64x32   41072   40.109     0.919
      2 128x64   47824   46.703     0.909
      1 128x64   86224   84.203     0.899

picked stride=2 hidden=16: ~5.0KB int8, test acc 0.9091 (best on grid 0.9192)


## TensorFlow Lite conversion (ESP32)

Float32 TFLite for reference, and full int8 quantization (weights + activations) which is what an ESP32 / TFLite Micro should run. The grid winner is converted with its own (pooled) representative data.

The scaler lives outside the model, so its parameters are saved separately for the firmware.

In [26]:
def to_tflite(src, int8, rep):
    conv = tf.lite.TFLiteConverter.from_keras_model(src)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = lambda: ([x[None]] for x in rep)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
    return conv.convert()


def tflite_accuracy(blob, X, y):
    it = tf.lite.Interpreter(model_content=blob)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    hits = 0
    for x, label in zip(X, y):
        if inp["dtype"] == np.int8:                      # quantize the input like the MCU will
            s, z = inp["quantization"]
            x = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
        it.set_tensor(inp["index"], x[None].astype(inp["dtype"]))
        it.invoke()
        hits += int(it.get_tensor(out["index"])[0].argmax() == label)  # scale > 0, so argmax survives
    return hits / len(y)


exports = {
    "model_float32.tflite": (to_tflite(model, False, X_train), X_test),
    "model_int8.tflite": (to_tflite(model, True, X_train), X_test),
    "model_best_int8.tflite": (to_tflite(best_model, True, best_X_train), best_X_test),
}
for name, (blob, Xte) in exports.items():
    open(name, "wb").write(blob)
    print(f"{name:26s} {len(blob) / 1024:6.1f}KB  test acc {tflite_accuracy(blob, Xte, y_test):.4f}")

# firmware must normalise raw sensor rows with the same scaler before inference
np.savez("scaler.npz", mean=scaler.mean_, scale=scaler.scale_)

INFO:tensorflow:Assets written to: /tmp/tmpg9dc189i/assets


INFO:tensorflow:Assets written to: /tmp/tmpg9dc189i/assets


Saved artifact at '/tmp/tmpg9dc189i'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 16), dtype=tf.float32, name=None)
Captures:
  140320616262272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140320609935648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140320614514224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140320614507504: TensorSpec(shape=(), dtype=tf.resource, name=None)
INFO:tensorflow:Assets written to: /tmp/tmpbj_eogzs/assets


W0000 00:00:1794948528.619455   63787 tf_tfl_flatbuffer_helpers.cc:425] Ignored output_format.
W0000 00:00:1794948528.619467   63787 tf_tfl_flatbuffer_helpers.cc:428] Ignored drop_control_dependency.
I0000 00:00:1794948528.619654   63787 reader.cc:83] Reading SavedModel from: /tmp/tmpg9dc189i
I0000 00:00:1794948528.619829   63787 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794948528.619833   63787 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpg9dc189i
I0000 00:00:1794948528.621260   63787 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1794948528.621458   63787 loader.cc:237] Restoring SavedModel bundle.
I0000 00:00:1794948528.631615   63787 loader.cc:221] Running initialization op on SavedModel bundle at path: /tmp/tmpg9dc189i
I0000 00:00:1794948528.634575   63787 loader.cc:484] SavedModel load for tags { serve }; Status: success: OK. Took 14925 microseconds.
I0000 00:00:1794948528.638571   63787

Saved artifact at '/tmp/tmpbj_eogzs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 16), dtype=tf.float32, name=None)
Captures:
  140320616262272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140320609935648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140320614514224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140320614507504: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/jordi/MERIT/IOTSS/omniflux/Train/enviro/lib/python3.14/site-packages/tensorflow/lite/python/convert.py:960: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1794948528.741272   63787 tf_tfl_flatbuffer_helpers.cc:425] Ignored output_format.
W0000 00:00:1794948528.741283   63787 tf_tfl_flatbuffer_helpers.cc:428] Ignored drop_control_dependency.
I0000 00:00:1794948528.741361   63787 reader.cc:83] Reading SavedModel from: /tmp/tmpbj_eogzs
I0000 00:00:1794948528.741510   63787 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794948528.741520   63787 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpbj_eogzs
I0000 00:00:1794948528.742705   63787 loader.cc:237] Restoring SavedModel bundle.
I0000 00:00:1794948528.751168   63787 loader.cc:221] Running initialization op on SavedModel bundle at path: /tmp/tmpbj_eogzs
I0000 00:00:1794948528.753768   63787 loader.cc:484] SavedM

... (2 outputs left out)

Saved artifact at '/tmp/tmpjvvfxr74'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 300), dtype=tf.float32, name='keras_tensor_35')
Output Type:
  TensorSpec(shape=(None, 16), dtype=tf.float32, name=None)
Captures:
  140317288547344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140317288555024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140317288552864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140316216929856: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/jordi/MERIT/IOTSS/omniflux/Train/enviro/lib/python3.14/site-packages/tensorflow/lite/python/convert.py:960: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


model_float32.tflite        194.6KB  test acc 0.8889
model_int8.tflite            52.7KB  test acc 0.8687
model_best_int8.tflite        7.8KB  test acc 0.9091


W0000 00:00:1794948528.861534   63787 tf_tfl_flatbuffer_helpers.cc:425] Ignored output_format.
W0000 00:00:1794948528.861546   63787 tf_tfl_flatbuffer_helpers.cc:428] Ignored drop_control_dependency.
I0000 00:00:1794948528.861623   63787 reader.cc:83] Reading SavedModel from: /tmp/tmpjvvfxr74
I0000 00:00:1794948528.861759   63787 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794948528.861762   63787 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpjvvfxr74
I0000 00:00:1794948528.862985   63787 loader.cc:237] Restoring SavedModel bundle.
I0000 00:00:1794948528.871332   63787 loader.cc:221] Running initialization op on SavedModel bundle at path: /tmp/tmpjvvfxr74
I0000 00:00:1794948528.873934   63787 loader.cc:484] SavedModel load for tags { serve }; Status: success: OK. Took 12314 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1794948528.907156   63787 flatbuffer_export.cc: